<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇪🇸 Spain Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: nap.transportes.gob.es · Provider: RENFE Operadora · Format: GTFS only · NeTEx not available
  </p>
</div>

## Data Source

**GTFS — RENFE Media, Larga Distancia y AVE**
- Source: Spanish National Access Point (NAP), operated by the Ministry
  of Transport (`nap.transportes.gob.es`)
- Dataset page: https://nap.transportes.gob.es/Files/Detail/897
- File downloaded: GTFS-ZIP
- Coverage: Renfe medium-distance, long-distance and high-speed (AVE)
  rail services across Spain
- Updated: 17/05/2026
- Registration required — a free account must be created on the NAP portal
  before downloading. Name, email, organisation and job title are required.
  Access is granted immediately after registration.

**NeTEx — not available**

No NeTEx dataset could be found for Spain through any official source.
The Spanish NAP does not offer NeTEx as a download format, only GTFS-ZIP,
GTFS-RT and SIRI are available. Neither Renfe nor any other Spanish operator
publishes NeTEx data publicly.

This is despite the EU regulation (Delegated Regulation 2017/1926) requiring
member states to publish multimodal travel information data in NeTEx format
through their National Access Points. Spain's non-compliance with this
requirement is itself a finding of this thesis, and will be discussed in
the overall conclusions.

For this reason, the GTFS–NeTEx MVD comparison cannot be performed for Spain.
The GTFS dataset is explored in Part 1 to document the data structure and
quality, and the absence of NeTEx is noted as a data availability finding.

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
from collections import defaultdict

In [1]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "20260517_000004_RENFE_AVLD.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/20260517_000004_RENFE_AVLD.zip


In [2]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
5,stop_times.txt,18.08
1,calendar_dates.txt,15.20
2,calendar.txt,2.23
6,trips.txt,2.23
4,stops.txt,0.34
3,routes.txt,0.23
0,agency.txt,0.00


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [4]:
# Load core GTFS tables
stops          = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes         = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips          = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

# Renfe's GTFS export pads the last column name in every file with trailing
# whitespace (e.g. "wheelchair_boarding                    "). Strip all
# column names here, once, right after loading, instead of only where it
# happens to matter downstream.
for _df in (stops, routes, trips, calendar, calendar_dates):
    _df.columns = _df.columns.str.strip()

print("stops:",          stops.shape)
print("routes:",         routes.shape)
print("trips:",          trips.shape)
print("calendar:", calendar.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (1019, 12)
routes: (681, 9)
trips: (6663, 9)
calendar: (6663, 10)
calendar_dates: (45401, 3)


In [5]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar columns:")
print(list(calendar.columns))
display(calendar.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

stops columns:
['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station', 'stop_timezone', 'wheelchair_boarding']


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,17000,NaN,Madrid-Chamartín-Clara Campoamor,NaN,40.471179,-3.682952,NaN,NaN,NaN,NaN,Europe/Madrid,1
1,37606,NaN,Badajoz,NaN,38.890696,-6.981753,NaN,NaN,NaN,NaN,Europe/Madrid,1
2,37603,NaN,Montijo,NaN,38.913539,-6.598577,NaN,NaN,NaN,NaN,Europe/Madrid,2
3,37500,NaN,Mérida,NaN,38.921503,-6.343794,NaN,NaN,NaN,NaN,Europe/Madrid,1
4,35400,NaN,Cáceres,NaN,39.461131,-6.385679,NaN,NaN,NaN,NaN,Europe/Madrid,1


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color']


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,1700037606GL023,1071,ALVIA,NaN,NaN,2,NaN,F2F5F5,...
1,1700056312GL026,1071,Intercity,NaN,NaN,2,NaN,F2F5F5,...
2,1700050300GL026,1071,Intercity,NaN,NaN,2,NaN,F2F5F5,...
3,5631217000GL026,1071,Intercity,NaN,NaN,2,NaN,F2F5F5,...
4,5030017000GL026,1071,Intercity,NaN,NaN,2,NaN,F2F5F5,...


trips columns:
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible']


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible
0,1700037606GL023,2026-05-172026-08-04001901,0019012026-05-17,NaN,190,NaN,NaN,NaN,1
1,1700037606GL023,2026-08-052026-10-11001901,0019012026-08-05,NaN,190,NaN,NaN,NaN,1
2,1700037606GL023,2026-10-132026-10-14001901,0019012026-10-13,NaN,190,NaN,NaN,NaN,1
3,1700037606GL023,2026-10-152026-10-31001901,0019012026-10-15,NaN,190,NaN,NaN,NaN,1
4,1700037606GL023,2026-11-032026-11-03001901,0019012026-11-03,NaN,190,NaN,NaN,NaN,1


calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,2026-05-172026-08-04001901,1,1,1,1,1,1,1,20260517,20260804
1,2026-08-052026-10-11001901,1,1,1,1,1,1,1,20260805,20261011
2,2026-10-132026-10-14001901,1,1,1,1,1,1,1,20261013,20261014
3,2026-10-152026-10-31001901,1,1,1,1,1,1,1,20261015,20261031
4,2026-11-032026-11-03001901,1,1,1,1,1,1,1,20261103,20261103


calendar_dates columns:
['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,2026-05-172026-08-04001901,20260517,2
1,2026-05-172026-08-04001901,20260523,2
2,2026-05-172026-08-04001901,20260524,2
3,2026-05-172026-08-04001901,20260530,2
4,2026-05-172026-08-04001901,20260531,2


## Comment 

**File structure:**
- The feed contains the standard GTFS files: `stop_times.txt` is the largest
  at 18.08 MB, followed by `calendar_dates.txt` at 15.20 MB
- Unlike Italy, Spain has **both `calendar.txt` and `calendar_dates.txt`**,
  meaning the calendar uses weekly patterns combined with exception dates

**Stops:**
- Stops have coordinates but no `stop_code`, `location_type` or `parent_station`, it has a flat
  structure, same as Italy
- Sample stops include major Spanish rail stations: Madrid Chamartín, Badajoz,
  Mérida, Cáceres, confirming national coverage
- Each stop has a `timezone` field set to `Europe/Madrid`

**Routes:**
- Route type 2 (Rail), consistent with a national rail feed
- Route short names include service types like **ALVIA**, Renfe uses named
  service brands similar to Trenitalia's FR, IC, REG categories

**Calendar:**
- Both `calendar.txt` and `calendar_dates.txt` are present
- `calendar.txt` shows weekly patterns with all 7 days set to 1 for some
  services, some services run every day
- `calendar_dates.txt` contains exception type 2 rows (removed dates), some
  services have specific days removed from the weekly pattern
- Feed validity runs from **2026-05-16 to 2026-11-03**

## Prepare GTFS station table

Since the Spanish GTFS feed has no `location_type` or `parent_station` columns,
all stops are used directly as the station-level table, the same approach used
for Italy.

In [6]:
# Prepare GTFS station table for Spain
gtfs_stations = stops.copy()
gtfs_stations = gtfs_stations.rename(columns={
    "stop_id":   "gtfs_stop_id",
    "stop_name": "gtfs_stop_name",
    "stop_lat":  "gtfs_lat",
    "stop_lon":  "gtfs_lon"
})

gtfs_station_quality = pd.DataFrame({
    "metric": [
        "total station rows",
        "rows with name",
        "rows missing name",
        "rows with coordinates",
        "rows missing coordinates",
        "unique station IDs",
        "duplicate station IDs"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["gtfs_stop_name"].notna().sum(),
        gtfs_stations["gtfs_stop_name"].isna().sum(),
        gtfs_stations[["gtfs_lat", "gtfs_lon"]].notna().all(axis=1).sum(),
        gtfs_stations[["gtfs_lat", "gtfs_lon"]].isna().any(axis=1).sum(),
        gtfs_stations["gtfs_stop_id"].nunique(),
        gtfs_stations["gtfs_stop_id"].duplicated().sum()
    ]
})
display(gtfs_station_quality)
display(gtfs_stations.head(10))

,metric,value
0,total station rows,1019
1,rows with name,1019
2,rows missing name,0
3,rows with coordinates,1019
4,rows missing coordinates,0
5,unique station IDs,1019
6,duplicate station IDs,0


,gtfs_stop_id,stop_code,gtfs_stop_name,stop_desc,gtfs_lat,gtfs_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,17000,NaN,Madrid-Chamartín-Clara Campoamor,NaN,40.471179,-3.682952,NaN,NaN,NaN,NaN,Europe/Madrid,1
1,37606,NaN,Badajoz,NaN,38.890696,-6.981753,NaN,NaN,NaN,NaN,Europe/Madrid,1
2,37603,NaN,Montijo,NaN,38.913539,-6.598577,NaN,NaN,NaN,NaN,Europe/Madrid,2
3,37500,NaN,Mérida,NaN,38.921503,-6.343794,NaN,NaN,NaN,NaN,Europe/Madrid,1
4,35400,NaN,Cáceres,NaN,39.461131,-6.385679,NaN,NaN,NaN,NaN,Europe/Madrid,1
5,30000,NaN,Monfragüe,NaN,39.937464,-6.100910,NaN,NaN,NaN,NaN,Europe/Madrid,2
6,35206,NaN,Navalmoral de la Mata,NaN,39.894913,-5.545607,NaN,NaN,NaN,NaN,Europe/Madrid,2
7,35203,NaN,Oropesa de Toledo,NaN,39.921746,-5.186137,NaN,NaN,NaN,NaN,Europe/Madrid,2
8,35200,NaN,Talavera de la Reina,NaN,39.970674,-4.826512,NaN,NaN,NaN,NaN,Europe/Madrid,2
9,35105,NaN,Torrijos,NaN,39.979003,-4.283037,NaN,NaN,NaN,NaN,Europe/Madrid,2


## Comment

The GTFS station table is clean and complete:

- **1,019 stops** all with names and coordinates, no missing values,
  all IDs unique
- `location_type` and `parent_station` columns are present but entirely
  empty (all NaN) confirming the flat structure with no stop hierarchy
- The sample confirms national coverage with stops like Madrid Chamartín, Badajoz,
  Mérida, Cáceres, Talavera de la Reina are spread across Spain
- Each stop has a `wheelchair_boarding` value (1 or 2). Accessibility
  information is included, which is more than most other countries provided
- `stop_timezone` is set to `Europe/Madrid` for all stops

## Prepare GTFS line table

In [7]:
# Prepare GTFS line table for Spain
gtfs_lines = routes.copy()

# Check route_short_name availability
print("route_short_name missing:", routes["route_short_name"].isna().sum())
print("route_long_name missing:", routes["route_long_name"].isna().sum())

# Use route_short_name as label, route_long_name as fallback
gtfs_lines["gtfs_line_label"] = gtfs_lines["route_short_name"].fillna(
    gtfs_lines["route_long_name"]
)

gtfs_lines = gtfs_lines.rename(columns={
    "route_id":   "gtfs_route_id",
    "route_type": "gtfs_route_type"
})[["gtfs_route_id", "gtfs_line_label", "gtfs_route_type"]]

gtfs_line_quality = pd.DataFrame({
    "metric": [
        "total line rows",
        "rows with line label",
        "rows missing line label",
        "unique line labels",
        "duplicate line labels"
    ],
    "value": [
        len(gtfs_lines),
        gtfs_lines["gtfs_line_label"].notna().sum(),
        gtfs_lines["gtfs_line_label"].isna().sum(),
        gtfs_lines["gtfs_line_label"].nunique(),
        gtfs_lines["gtfs_line_label"].duplicated().sum()
    ]
})
display(gtfs_line_quality)

print("route_type distribution:")
display(routes["route_type"].value_counts(dropna=False).reset_index())

print("Sample routes:")
display(routes[["route_id", "route_short_name", "route_long_name", "route_type"]].head(15))

route_short_name missing: 0
route_long_name missing: 681


,metric,value
0,total line rows,681
1,rows with line label,681
2,rows missing line label,0
3,unique line labels,13
4,duplicate line labels,668


route_type distribution:


,route_type,count
0,2,681


Sample routes:


,route_id,route_short_name,route_long_name,route_type
0,1700037606GL023,ALVIA,NaN,2
1,1700056312GL026,Intercity,NaN,2
2,1700050300GL026,Intercity,NaN,2
3,5631217000GL026,Intercity,NaN,2
4,5030017000GL026,Intercity,NaN,2
5,1120013200GL026,Intercity,NaN,2
6,1151111208GL026,Intercity,NaN,2
7,1151111200GL026,Intercity,NaN,2
8,1120811511GL026,Intercity,NaN,2
9,1120011511GL026,Intercity,NaN,2


## Comment

The routes table reveals an interesting structure for the Spanish GTFS feed:

- **681 routes** in total, all rail (`route_type = 2`)
- **All 681 have a `route_short_name`** there are no missing values, no fallback needed
- Only **13 unique line labels** despite 681 routes, meaning the same service
  brand covers many individual routes. For example "Intercity" appears hundreds
  of times as different route IDs but the same label
- **668 duplicate labels**. Renfe uses service brand names
  (ALVIA, Intercity, TRENCELTA) as the short name, not a unique route identifier
- `route_long_name` is missing for all 681 routes. Labels come entirely from
  `route_short_name`
- Service brands visible in the sample: **ALVIA**, **Intercity**, **TRENCELTA**,
    the same category-based approach as Italy's NeTEx, but here in GTFS

This is very similar to what was found on the NeTEx side for Italy.  Renfe
organises its GTFS lines by service type rather than geographic route, which
is unusual for GTFS but consistent with how national rail operators tend to
think about their services.

## Prepare GTFS calendar

Spain uses both `calendar.txt` and `calendar_dates.txt` to define when
services run. The active dates are rebuilt in three steps:

1. For each service in `calendar.txt`, the weekly pattern (which days of
   the week it runs) is applied across the validity period to generate
   all active dates
2. Exception dates from `calendar_dates.txt` are then applied. Type 1
   rows add specific dates and type 2 rows remove them
3. Services that only appear in `calendar_dates.txt` (with no weekly
   pattern) get their active dates purely from the added exception dates

The result is a table of one row per service ID per active date, which
would have been used for the calendar comparison in Part 3.

In [8]:
# Convert dates
calendar["start_date_dt"] = pd.to_datetime(calendar["start_date"], format="%Y%m%d")
calendar["end_date_dt"]   = pd.to_datetime(calendar["end_date"],   format="%Y%m%d")
calendar_dates["date_dt"] = pd.to_datetime(calendar_dates["date"], format="%Y%m%d")

weekday_cols = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday"
]

comparison_start = min(calendar["start_date_dt"].min(), calendar_dates["date_dt"].min())
comparison_end   = max(calendar["end_date_dt"].max(),   calendar_dates["date_dt"].max())

print(f"Feed window: {comparison_start.date()} to {comparison_end.date()}")

# Build added and removed dates from calendar_dates.txt
calendar_dates["service_id"] = calendar_dates["service_id"].astype(str)
added_dates   = defaultdict(set)
removed_dates = defaultdict(set)

for _, row in calendar_dates.iterrows():
    if row["exception_type"] == 1:
        added_dates[row["service_id"]].add(row["date_dt"])
    elif row["exception_type"] == 2:
        removed_dates[row["service_id"]].add(row["date_dt"])

# Rebuild active dates
calendar["service_id"] = calendar["service_id"].astype(str)
gtfs_active_date_rows = []

for _, row in calendar.iterrows():
    sid   = row["service_id"]
    start = max(row["start_date_dt"], comparison_start)
    end   = min(row["end_date_dt"],   comparison_end)
    if start <= end:
        all_days     = pd.date_range(start, end, freq="D")
        active_dates = set()
        for d in all_days:
            if row[weekday_cols[d.weekday()]] == 1:
                active_dates.add(d)
        active_dates = (active_dates - removed_dates[sid]) | {
            d for d in added_dates[sid]
            if comparison_start <= d <= comparison_end
        }
        for d in sorted(active_dates):
            gtfs_active_date_rows.append({"service_id": sid, "active_date": d})

# Add calendar_dates-only service IDs
calendar_only_ids = set(calendar["service_id"])
date_only_ids     = set(calendar_dates["service_id"]) - calendar_only_ids
for sid in date_only_ids:
    for d in sorted(added_dates[sid]):
        if comparison_start <= d <= comparison_end:
            gtfs_active_date_rows.append({"service_id": sid, "active_date": d})

gtfs_active_dates = pd.DataFrame(gtfs_active_date_rows)
print(f"Active dates built for {gtfs_active_dates['service_id'].nunique()} service IDs.")
print(f"Total active date rows: {len(gtfs_active_dates)}")
display(gtfs_active_dates.head(10))

Feed window: 2026-05-16 to 2026-11-03
Active dates built for 6663 service IDs.
Total active date rows: 156324


,service_id,active_date
0,2026-05-172026-08-04001901,2026-05-18
1,2026-05-172026-08-04001901,2026-05-19
2,2026-05-172026-08-04001901,2026-05-20
3,2026-05-172026-08-04001901,2026-05-21
4,2026-05-172026-08-04001901,2026-05-22
5,2026-05-172026-08-04001901,2026-05-25
6,2026-05-172026-08-04001901,2026-05-26
7,2026-05-172026-08-04001901,2026-05-27
8,2026-05-172026-08-04001901,2026-05-28
9,2026-05-172026-08-04001901,2026-05-29


## Comment

The active dates were built successfully:

- **6,663 service IDs** with active dates consistent with the number of
  trips in the feed
- **156,324 total active date rows** covering the feed window from
  2026-05-16 to 2026-11-03
- The service IDs follow the format `2026-05-172026-08-04001901`, a
  concatenation of the start date, end date and a numeric code. This is
  an unusual format compared to other countries but works correctly
- The sample shows weekday services running Monday to Friday with weekends
  skipped (gaps between 22 May and 25 May), consistent with the weekly
  pattern in `calendar.txt`


## Conclusion

Spain is one of the countries where the GTFS–NeTEx comparison could not be
performed because no publicly available NeTEx dataset was found. Belgium is
another example in this analysis, where NeTEx exists but is restricted behind a license
agreement with SNCB.

The GTFS feed for RENFE Media, Larga Distancia y AVE is well-structured and
complete: 1,019 stops, 681 routes and 6,663 service IDs, all with names,
coordinates and calendar data. The feed covers Renfe medium-distance,
long-distance and high-speed rail services across Spain, and is published
through Spain's NAP with free registration.

However, despite the EU Delegated Regulation 2017/1926 requiring member states
to publish multimodal travel data in NeTEx format through their National Access
Points, no NeTEx dataset could be found for Spain. The Spanish NAP only offers
GTFS-ZIP, GTFS-RT and SIRI formats. Neither Renfe nor any other Spanish operator
publishes NeTEx data publicly.

Belgium and Spain together show that NeTEx availability in Europe is still
uneven: Belgium restricts access through licensing, while in the sources
checked, no publicly available Spanish NeTEx dataset could be found. Both
cases prevent a meaningful comparison and this will be mentioned in the overall conclusions of the thesis.